# Everpure CLV Predictive Analytics
## 01 — Dataset Understanding

Objective:
Understand the dataset structure, business variables,
and potential prediction targets before preprocessing.

Dataset: Case Study 2 — CLV Predictive Analytics

In [1]:
import pandas as pd
from pathlib import Path

# Path to the original dataset
DATA_PATH = Path("../data/raw/Case_Study_2_CLV_Predictive_Analytics.csv")

# Load CSV into a pandas DataFrame
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [2]:
# Dataset dimensions
print("Dataset shape:", df.shape)

# Number of records
print("Total rows:", df.shape[0])

# Number of variables
print("Total columns:", df.shape[1])

# Display column names
print("\nColumn names:")
for col in df.columns:
    print("-", col)

Dataset shape: (2000, 9)
Total rows: 2000
Total columns: 9

Column names:
- account_id
- array_id
- contract_end_date
- historical_15yr_net_profit_usd
- current_capacity_tb
- utilization_rate_pct
- static_baseline_churn_rate
- churn_event_next_12m
- expansion_potential_signal


In [3]:
# Display the first 5 records
df.head()

,account_id,array_id,contract_end_date,historical_15yr_net_profit_usd,current_capacity_tb,utilization_rate_pct,static_baseline_churn_rate,churn_event_next_12m,expansion_potential_signal
0,ACC_1000,ARR_5000,2026-10-01,146797,1382,0.59,0.05,0.0,Medium
1,ACC_1001,ARR_5001,2026-10-01,1236408,1809,0.93,0.05,1.0,UNKNOWN
2,ACC_1002,ARR_5002,2026-10-02,2790632,605,1.28,0.05,0.0,Low
3,ACC_1003,ARR_5003,2026-10-02,1233288,1567,0.47,0.05,0.0,Low
4,ACC_1004,ARR_5004,2026-10-03,2881167,488,1.22,0.05,0.0,Low


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   account_id                      2000 non-null   str    
 1   array_id                        2000 non-null   str    
 2   contract_end_date               2000 non-null   str    
 3   historical_15yr_net_profit_usd  2000 non-null   int64  
 4   current_capacity_tb             2000 non-null   int64  
 5   utilization_rate_pct            2000 non-null   float64
 6   static_baseline_churn_rate      2000 non-null   float64
 7   churn_event_next_12m            1884 non-null   float64
 8   expansion_potential_signal      2000 non-null   str    
dtypes: float64(3), int64(2), str(4)
memory usage: 200.1 KB


In [6]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2)
})

display(missing_summary)

,Missing Count,Missing %
account_id,0,0.0
array_id,0,0.0
contract_end_date,0,0.0
historical_15yr_net_profit_usd,0,0.0
current_capacity_tb,0,0.0
utilization_rate_pct,0,0.0
static_baseline_churn_rate,0,0.0
churn_event_next_12m,116,5.8
expansion_potential_signal,0,0.0


In [7]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

account_id: 1000 unique values
array_id: 2000 unique values
contract_end_date: 1000 unique values
historical_15yr_net_profit_usd: 2000 unique values
current_capacity_tb: 1366 unique values
utilization_rate_pct: 111 unique values
static_baseline_churn_rate: 1 unique values
churn_event_next_12m: 2 unique values
expansion_potential_signal: 4 unique values


In [8]:
print("Churn label distribution:")
print(df["churn_event_next_12m"].value_counts(dropna=False))

print("\nExpansion signal distribution:")
print(df["expansion_potential_signal"].value_counts(dropna=False))

Churn label distribution:
churn_event_next_12m
0.0    1586
1.0     298
NaN     116
Name: count, dtype: int64

Expansion signal distribution:
expansion_potential_signal
Low        841
Medium     622
High       426
UNKNOWN    111
Name: count, dtype: int64


### Prediction Target and Features

Target:
- churn_event_next_12m

Potential predictive features:
- contract_end_date
- historical_15yr_net_profit_usd
- current_capacity_tb
- utilization_rate_pct
- expansion_potential_signal

Identifiers:
- account_id
- array_id

Existing baseline:
- static_baseline_churn_rate

Important:
Feature availability and leakage risks must be checked before modeling.

In [9]:
print("Total records:", len(df))

print("Unique accounts:", df["account_id"].nunique())

print("Unique arrays:", df["array_id"].nunique())

print("\nArrays per account:")
print(df.groupby("account_id")["array_id"].nunique().describe())

print("\nExample accounts:")
display(
    df.groupby("account_id")["array_id"]
      .nunique()
      .head(10)
)

Total records: 2000
Unique accounts: 1000
Unique arrays: 2000

Arrays per account:
count    1000.0
mean        2.0
std         0.0
min         2.0
25%         2.0
50%         2.0
75%         2.0
max         2.0
Name: array_id, dtype: float64

Example accounts:


account_id
ACC_1000    2
ACC_1001    2
ACC_1002    2
ACC_1003    2
ACC_1004    2
ACC_1005    2
ACC_1006    2
ACC_1007    2
ACC_1008    2
ACC_1009    2
Name: array_id, dtype: int64

In [10]:
account_churn = (
    df.groupby("account_id")["churn_event_next_12m"]
      .agg(
          labeled_rows="count",
          distinct_labels=lambda x: x.dropna().nunique()
      )
)

print("Accounts with conflicting observed churn labels:")
print((account_churn["distinct_labels"] > 1).sum())

display(account_churn.head(10))

Accounts with conflicting observed churn labels:
224


,labeled_rows,distinct_labels
account_id,,
ACC_1000,2,2
ACC_1001,2,2
ACC_1002,2,2
ACC_1003,2,1
ACC_1004,1,1
ACC_1005,2,1
ACC_1006,1,1
ACC_1007,2,1
ACC_1008,2,1


In [11]:
feature_roles = pd.DataFrame({
    "Column": df.columns,
    "Role": [
        "Account identifier",
        "Array identifier",
        "Potential feature - date",
        "Potential feature - leakage review",
        "Potential feature - capacity",
        "Potential feature - utilization",
        "Baseline only",
        "Target",
        "Potential feature - categorical"
    ]
})

display(feature_roles)

,Column,Role
0,account_id,Account identifier
1,array_id,Array identifier
2,contract_end_date,Potential feature - date
3,historical_15yr_net_profit_usd,Potential feature - leakage review
4,current_capacity_tb,Potential feature - capacity
5,utilization_rate_pct,Potential feature - utilization
6,static_baseline_churn_rate,Baseline only
7,churn_event_next_12m,Target
8,expansion_potential_signal,Potential feature - categorical


## Initial Findings — Account and Array Structure

1. The dataset contains 2,000 storage-array records associated
   with 1,000 unique customer accounts.

2. Every account has exactly two arrays.

3. There are 224 accounts with differing observed churn labels
   across their arrays.

4. This distinction suggests that an array churn event cannot
   automatically be interpreted as complete customer churn.

5. To reduce data leakage, all arrays belonging to the same
   account will remain in the same training, validation, or
   test partition.

6. The primary prediction target is array-level churn within
   the next 12 months.

7. The definition of account-level churn requires additional
   business information about customer relationships and
   array retirement events.

In [12]:
print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
print(df.isna().sum())

print("\nTARGET DISTRIBUTION")
print(df["churn_event_next_12m"].value_counts(dropna=False))

print("\nEXPANSION SIGNAL DISTRIBUTION")
print(df["expansion_potential_signal"].value_counts(dropna=False))

DATA TYPES
account_id                            str
array_id                              str
contract_end_date                     str
historical_15yr_net_profit_usd      int64
current_capacity_tb                 int64
utilization_rate_pct              float64
static_baseline_churn_rate        float64
churn_event_next_12m              float64
expansion_potential_signal            str
dtype: object

MISSING VALUES
account_id                          0
array_id                            0
contract_end_date                   0
historical_15yr_net_profit_usd      0
current_capacity_tb                 0
utilization_rate_pct                0
static_baseline_churn_rate          0
churn_event_next_12m              116
expansion_potential_signal          0
dtype: int64

TARGET DISTRIBUTION
churn_event_next_12m
0.0    1586
1.0     298
NaN     116
Name: count, dtype: int64

EXPANSION SIGNAL DISTRIBUTION
expansion_potential_signal
Low        841
Medium     622
High       426
UNKNOWN    111
Name

## 1.3 — Data Dictionary and Problem Definition

In [13]:
data_dictionary = pd.DataFrame({
    "Column": [
        "account_id",
        "array_id",
        "contract_end_date",
        "historical_15yr_net_profit_usd",
        "current_capacity_tb",
        "utilization_rate_pct",
        "static_baseline_churn_rate",
        "churn_event_next_12m",
        "expansion_potential_signal"
    ],
    "Business Meaning": [
        "Customer account identifier",
        "Storage array identifier",
        "Contract expiration date",
        "Reported historical 15-year net profit in USD",
        "Current storage capacity in terabytes",
        "Reported storage utilization",
        "Existing assumed churn probability",
        "Observed churn event within next 12 months",
        "Business signal indicating expansion potential"
    ],
    "Role": [
        "Grouping identifier",
        "Record identifier",
        "Potential feature",
        "Potential feature - leakage review",
        "Predictive feature",
        "Potential feature - unit validation",
        "Baseline only",
        "Target",
        "Categorical feature"
    ]
})

display(data_dictionary)

,Column,Business Meaning,Role
0,account_id,Customer account identifier,Grouping identifier
1,array_id,Storage array identifier,Record identifier
2,contract_end_date,Contract expiration date,Potential feature
3,historical_15yr_net_profit_usd,Reported historical 15-year net profit in USD,Potential feature - leakage review
4,current_capacity_tb,Current storage capacity in terabytes,Predictive feature
5,utilization_rate_pct,Reported storage utilization,Potential feature - unit validation
6,static_baseline_churn_rate,Existing assumed churn probability,Baseline only
7,churn_event_next_12m,Observed churn event within next 12 months,Target
8,expansion_potential_signal,Business signal indicating expansion potential,Categorical feature


### Primary Prediction Target

The objective is to estimate the probability of an
array-level churn event within the next 12 months.

Target column: churn_event_next_12m

- 1 = Churn event observed
- 0 = No churn event observed
- Missing = Outcome unavailable or unknown

The existing static_baseline_churn_rate of 0.05
will be used as the benchmark.

The primary output will be a calibrated 12-month
churn probability for each storage array.

In [14]:
TARGET = "churn_event_next_12m"
GROUP_COLUMN = "account_id"
BASELINE_COLUMN = "static_baseline_churn_rate"

CORE_FEATURES = [
    "current_capacity_tb",
    "utilization_rate_pct",
    "expansion_potential_signal"
]

FEATURES_REQUIRING_REVIEW = [
    "contract_end_date",
    "historical_15yr_net_profit_usd"
]

print("Target:", TARGET)
print("Grouping variable:", GROUP_COLUMN)
print("Baseline:", BASELINE_COLUMN)
print("Core features:", CORE_FEATURES)
print("Features requiring review:", FEATURES_REQUIRING_REVIEW)

Target: churn_event_next_12m
Grouping variable: account_id
Baseline: static_baseline_churn_rate
Core features: ['current_capacity_tb', 'utilization_rate_pct', 'expansion_potential_signal']
Features requiring review: ['contract_end_date', 'historical_15yr_net_profit_usd']


In [15]:
contract_dates = pd.to_datetime(
    df["contract_end_date"],
    errors="coerce"
)

print("Earliest contract end:", contract_dates.min())
print("Latest contract end:", contract_dates.max())
print("Invalid dates:", contract_dates.isna().sum())

print("\nContract end years:")
print(contract_dates.dt.year.value_counts().sort_index())


Earliest contract end: 2026-10-01 00:00:00
Latest contract end: 2029-06-26 00:00:00
Invalid dates: 0

Contract end years:
contract_end_date
2026    184
2027    730
2028    732
2029    354
Name: count, dtype: int64


### Initial Modeling Decisions

1. The primary prediction target is array-level
   churn within the next 12 months.

2. Customer accounts have multiple arrays. We will
   use account-grouped train/validation/test splits
   to avoid cross-account leakage.

3. The static 5% churn assumption is the baseline.

4. Current capacity, utilization, and expansion
   signal are provisional predictive features.

5. Contract dates and 15-year net profit require
   prediction-time availability checks.

6. Missing churn labels will not be treated as
   confirmed non-churn events.

7. Array-level churn is not equivalent to complete
   account-level churn.

8. The available data supports evaluation of
   12-month churn probabilities, but not a fully
   validated 15-year survival forecast.

### Contract Date Analysis

- Contract end dates range from 1 October 2026 to 26 June 2029.
- All 2,000 contract dates are valid.
- Most contracts end during 2027 and 2028.
- Contract expiration is not automatically equivalent to churn.
- A prediction reference date is required before calculating
  days until contract expiry.
- Historical scoring snapshots are unavailable, so genuine
  chronological back-testing cannot be established from
  this dataset alone.

## 1.4 — Business Context and Project Scope

### Business Problem
Everpure's customer lifetime value calculations rely partly on
assumed churn, retention, and expansion rates.

The objective of this case study is to investigate whether
machine learning can improve churn-risk estimation compared
with the existing static 5% churn assumption.

### Dataset
- 2,000 storage-array records
- 1,000 customer accounts
- Two arrays per account
- 12-month array-level churn target
- Historical profit, storage capacity, utilization,
  contract dates, and expansion signals

### Project Objectives
1. Evaluate the existing 5% churn baseline.
2. Train and compare Random Forest and XGBoost models.
3. Evaluate predictive accuracy and probability calibration.
4. Examine performance across customer segments.
5. Explain predictions using SHAP.
6. Write model scores to a Snowflake output table.

### Key Assumptions and Limitations
- Array-level churn does not necessarily represent complete
  customer loss.
- Missing churn labels are not assumed to mean retention.
- Historical feature availability must be verified to
  avoid target leakage.
- The dataset does not provide explicit historical
  prediction snapshots.
- A fully validated 15-year survival forecast cannot be
  established from the supplied 12-month outcome labels.
- Expansion potential is an input signal, not an observed
  expansion outcome.